# 01 — Análise exploratória (EDA) da base de vendas

Base oficial: `data/vendas_cliente.xlsx` (aba `Worksheet`), pedidos de certificações ANEC de 05/09/2024 a 07/04/2026.

**LGPD:** a planilha tem nomes, telefones e endereços reais. Este notebook **só mostra contagens e somas**; nenhuma linha individual é exibida e as colunas pessoais são descartadas logo após a leitura.

Convenções (CLAUDE.md, seção 5): venda = `Status Venda` = `pago`; data de referência = `Criado em`; B2B = `Conta` preenchida.

In [ ]:
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.io as pio

pio.renderers.default = "notebook_connected"
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CAMINHO = RAIZ / "data" / "vendas_cliente.xlsx"
bruto = pd.read_excel(CAMINHO, sheet_name="Worksheet")
print(f"{len(bruto):,} linhas × {bruto.shape[1]} colunas")

## 1. Preparação mínima (sem dados pessoais)

Guardamos só as colunas analíticas. `Conta` vira um indicador B2B/B2C (o nome da empresa não é mantido).

In [ ]:
COLUNAS_PESSOAIS = ["Comprador", "Nome Pagamento", "Telefone/Celular", "CEP",
                    "Logradouro", "Número.1", "Bairro", "Cidade", "Estado"]

def parse_brl(serie: pd.Series) -> pd.Series:
    """Converte texto 'R$1.234,56' em float."""
    texto = (serie.astype(str).str.replace("R$", "", regex=False).str.strip()
             .str.replace(".", "", regex=False).str.replace(",", ".", regex=False))
    return pd.to_numeric(texto, errors="coerce")

conta_preenchida = bruto["Conta"].notna() & (bruto["Conta"].astype(str).str.strip() != "")
df = bruto.drop(columns=COLUNAS_PESSOAIS + ["Conta"]).assign(
    canal=conta_preenchida.map({True: "B2B", False: "B2C"}),
    valor=parse_brl(bruto["Valor Pago"]),
    criado_em=pd.to_datetime(bruto["Criado em"], format="%d/%m/%Y %H:%M:%S", errors="coerce"),
)
del bruto  # libera a cópia com dados pessoais
print("Colunas mantidas:", list(df.columns))
print("Datas 'Criado em' inválidas:", df["criado_em"].isna().sum())
print("Período:", df["criado_em"].min(), "→", df["criado_em"].max())

## 2. Pedidos por `Status Venda`

In [ ]:
status = (df["Status Venda"].value_counts().rename("pedidos").to_frame()
          .assign(percentual=lambda t: 100 * t["pedidos"] / t["pedidos"].sum()))
status

## 3. Receita paga total

Valor de conferência esperado: **R$ 308.783,48** em 2.919 pedidos pagos.

In [ ]:
pagos = df[df["Status Venda"] == "pago"].copy()
receita_total = round(pagos["valor"].sum(), 2)
print(f"Pedidos pagos: {len(pagos):,}")
print(f"Receita paga total: R$ {receita_total:,.2f}")
print(f"Pedidos pagos com R$ 0,00: {(pagos['valor'] == 0).sum()}")
assert receita_total == 308783.48, receita_total

## 4. Pedidos pagos por mês (por `Criado em`)

Abril/2026 está incompleto (dados até o dia 7).

In [ ]:
pagos["mes"] = pagos["criado_em"].dt.to_period("M")
por_mes = (pagos.groupby("mes").agg(pedidos=("Número", "size"), receita=("valor", "sum"))
           .assign(ticket_medio=lambda t: t["receita"] / t["pedidos"]))
display(por_mes)

grafico = por_mes.reset_index().assign(mes=lambda t: t["mes"].astype(str))
px.bar(grafico, x="mes", y="pedidos", title="Pedidos pagos por mês (Criado em)",
       labels={"mes": "Mês", "pedidos": "Pedidos pagos"}).show()

## 5. Separação B2B × B2C (pedidos pagos)

Conferência: **82 pedidos pagos B2B somando R$ 88.410,08**.

In [ ]:
canal = (pagos.groupby("canal").agg(pedidos=("Número", "size"), receita=("valor", "sum"))
         .assign(ticket_medio=lambda t: t["receita"] / t["pedidos"],
                 perc_receita=lambda t: 100 * t["receita"] / t["receita"].sum()))
display(canal)

canal_mes = pagos.pivot_table(index="mes", columns="canal", values="Número",
                              aggfunc="size", fill_value=0)
display(canal_mes)
px.bar(canal_mes.reset_index().assign(mes=lambda t: t["mes"].astype(str)),
       x="mes", y=["B2C", "B2B"], title="Pedidos pagos por mês e canal",
       labels={"mes": "Mês", "value": "Pedidos pagos", "variable": "Canal"}).show()

Observação: antes de ago/2025 o volume é baixo e dominado por B2B; por isso a previsão usará só B2C a partir do primeiro mês com ≥ 100 pedidos pagos (CLAUDE.md, decisões 3 e 4).

## 6. Pedidos por dia da semana (por `Criado em`)

In [ ]:
DIAS = ["segunda", "terça", "quarta", "quinta", "sexta", "sábado", "domingo"]
semana = pd.DataFrame({
    "todos_pedidos": df["criado_em"].dt.dayofweek.value_counts(),
    "pedidos_pagos": pagos["criado_em"].dt.dayofweek.value_counts(),
}).sort_index().set_axis(DIAS).assign(
    taxa_pagamento=lambda t: 100 * t["pedidos_pagos"] / t["todos_pedidos"])
display(semana)
px.bar(semana.reset_index(names="dia"), x="dia", y=["todos_pedidos", "pedidos_pagos"],
       barmode="group", title="Pedidos por dia da semana",
       labels={"dia": "Dia", "value": "Pedidos", "variable": ""}).show()

## 7. Resumo

- A base tem 4.558 pedidos; 2.919 pagos, somando R$ 308.783,48.
- B2B são poucos pedidos (82 pagos) mas de ticket alto; B2C concentra o volume.
- O volume mensal muda de patamar a partir de ago/2025; abril/2026 está incompleto.
- Há padrão semanal claro, o que justifica sazonalidade de 7 dias nos modelos da Fase 2.